In [1]:
%pip install pandas scikit-learn
import pandas as pd


Note: you may need to restart the kernel to use updated packages.


In [2]:
df = pd.read_csv("data/postings.csv", usecols=['title','description','formatted_experience_level'])
df = df.dropna(subset= 'formatted_experience_level')

In [3]:
df['text'] = df['title'] + " " + df["description"]
y = df['formatted_experience_level']
x = df.text
x

70        Entry Level Oracle Financial Technology Consul...
84        Quality Assurance Manager Galerie is seeking a...
85        Validation Engineer, Labware LIMS Validation E...
101       Administrative Assistant - CONCUR Global Finan...
102       Seasonal Office Administrator Seasonal Office ...
                                ...                        
123843    Quality Engineer Position: Quality Engineer I ...
123844    Title IX/Investigations Attorney Our Walnut Cr...
123845    Staff Software Engineer, ML Serving Platform A...
123846    Account Executive, Oregon/Washington Company O...
123848    Marketing Social Media Specialist Marketing So...
Name: text, Length: 94440, dtype: object

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test =train_test_split(x,y, test_size = 0.2, random_state=42, stratify = y)

In [5]:
X_test

40222     Warehouse Associate - 2nd Shift Pick to Pallet...
109464    In-Store Shopper A Day in the Life:\n\nAs the ...
10808     LATAM and ROW Content Program Sr. Manager  BEP...
83751     Attorney - Employment Defense Litigation - Bou...
54690     LVN - Bluitt Flowers Health Center - Full Time...
                                ...                        
48352     Senior Project Controller (EPC Construction) C...
24983     Delivery Driver What’s Unique About You Is Wha...
83026     Shipping Inspector Coordinator SUMMARY: The Sh...
8308      Facilities - Mechanical Engineer - Aerospace R...
109538    03225 Inside Sales Overview\n\nSALLY BEAUTY AD...
Name: text, Length: 18888, dtype: object

In [6]:
df['text'].isna().sum()

np.int64(0)

In [7]:
y.value_counts()

formatted_experience_level
Mid-Senior level    41489
Entry level         36708
Associate            9826
Director             3746
Internship           1449
Executive            1222
Name: count, dtype: int64

In [8]:
y_test.value_counts()

formatted_experience_level
Mid-Senior level    8298
Entry level         7342
Associate           1965
Director             749
Internship           290
Executive            244
Name: count, dtype: int64

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

In [10]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter = 1000, class_weight = "balanced").fit(X_train_transformed, y_train)
clf.score(X_test_transformed, y_test)

0.7189220669207963

In [11]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
a = classification_report(y_test, clf.predict(X_test_transformed))
print(a)
b = confusion_matrix(y_test, clf.predict(X_test_transformed))
print(b)

                  precision    recall  f1-score   support

       Associate       0.43      0.69      0.53      1965
        Director       0.50      0.79      0.61       749
     Entry level       0.82      0.77      0.79      7342
       Executive       0.40      0.68      0.50       244
      Internship       0.66      0.90      0.76       290
Mid-Senior level       0.81      0.67      0.73      8298

        accuracy                           0.72     18888
       macro avg       0.60      0.75      0.65     18888
    weighted avg       0.75      0.72      0.73     18888

[[1349   48  197   18    6  347]
 [  16  590   15   44    4   80]
 [ 693   63 5664   26   79  817]
 [  10   27    6  166    1   34]
 [   2    2   19    1  260    6]
 [1063  457 1024  159   45 5550]]


In [15]:
title = df.loc[X_test.index, "title"]
title

40222     Warehouse Associate - 2nd Shift Pick to Pallet...
109464                                     In-Store Shopper
10808            LATAM and ROW Content Program Sr. Manager 
83751     Attorney - Employment Defense Litigation - Bou...
54690        LVN - Bluitt Flowers Health Center - Full Time
                                ...                        
48352          Senior Project Controller (EPC Construction)
24983                                       Delivery Driver
83026                        Shipping Inspector Coordinator
8308      Facilities - Mechanical Engineer - Aerospace R...
109538                                   03225 Inside Sales
Name: title, Length: 18888, dtype: object

In [16]:
pattern = r"\b(senior|sr|jr|junior|intern|internship|director|executive|chief|head|lead|entry|associte|principal|vp)\b"

In [34]:
has_tell = title.str.lower().str.contains(pattern, regex=True)
mask = ~has_tell
mask

C:\Users\Thuan\AppData\Local\Temp\ipykernel_11048\2150038629.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_tell = title.str.lower().str.contains(pattern, regex=True)


40222     False
109464     True
10808     False
83751      True
54690      True
          ...  
48352     False
24983      True
83026      True
8308       True
109538     True
Name: title, Length: 18888, dtype: bool

In [35]:
mask.sum()

np.int64(14450)

In [37]:
y_hard = y_test[mask]
X_hard = X_test_transformed[mask.values]

In [38]:
(y_hard == "Mid-Senior level").mean()
model.score(X_hard, y_hard)            

NameError: name 'model' is not defined